# Chapter 20 — Slash Commands and Custom UI

**Companion to *Pi Agents*** · [`Pi Agents` chapter 20](https://programmer.ie/books/pi/20-chapter/)

| | |
|---|---|
| Chapter | `20-chapter.md` · weight 20 · `/books/pi/20-chapter/` |
| Notebook | `notebooks/pi/20-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Are `hasUI` and `mode === "tui"` different guards?

The chapter's point: a command can work with a UI (RPC) and degrade without
one (print/JSON). A custom screen (`ctx.ui.custom`) exists **only** where a real
terminal is attached (`mode === "tui"`). The component API uses display columns,
not string length.


## What this notebook establishes

- `ctx.hasUI` is true for RPC mode (and TUI); false for print/JSON. A command can degrade gracefully.
- `ctx.mode === "tui"` is a **stricter guard** — custom screens (`ctx.ui.custom`) only exist in interactive terminal mode.
- The component API measures width in **display columns** (`visibleWidth`), not string length — wide characters count correctly.
- Component output is **cached by width** and recomputed only after `invalidate()` — re-laying-out on every keystroke is what makes a terminal feel slow.
- A component finishes through its completion callback on `q` or `Escape`.

## What this notebook does **not** establish

- **The custom screen and entry renderer were not drawn** — they compile against the pinned declarations and the component is tested directly (`ch20-lim1`).
- **No terminal rendering is exercised** — this is a headless run. Layer 1–2 checks (Shift+Enter, Kitty keyboard, tmux passthrough) are in chapter 44 (`ch20-lim2`).
- **`registerToolRenderer` is documented-not-run** by the suite (`ch17-lim3`).
- **The smallest command receives raw argument text** — no parsing, no completions unless you add them.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with the chapter's guard-ui and guard-minimal extensions, plus direct component tests. No terminal is attached; RPC mode is used where a UI is needed.
* **Evidence scope:** `in_process_runtime`. The command modes, component API, and `hasUI`/`mode` distinction are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(20))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/20-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## With a UI (RPC mode): the command opens a dialog

`/guard` with `ctx.hasUI` true calls `ctx.ui.select` and shows the answer as a
notification.

In [3]:
UI = pinb.driver_source("ch20-ui.ts")
ui = pinb.driver(UI, label="ch20-ui", timeout=300)
print(pinb.md_table(
    ["mode", "select title", "last notification"],
    [["RPC (hasUI)", ui["withUI"]["selectTitle"], ui["withUI"]["lastNotification"]]],
))

| mode | select title | last notification |
|---|---|---|
| RPC (hasUI) | Path guard | 0 blocked call(s) this session |


In [4]:
pinb.show_checks([
    pinb.check("with UI: /guard opens a select dialog",
               ui["withUI"]["selectTitle"] == "Path guard", "select opened"),
    pinb.check("the answer is shown as a notification",
               ui["withUI"]["lastNotification"] and "blocked call" in ui["withUI"]["lastNotification"], "notification shown"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------  ------------------
PASS  with UI: /guard opens a select dialog  select opened
PASS  the answer is shown as a notification  notification shown

2/2 assertions held.


## Without a UI (print/JSON mode): a minimal command works without dialogs

`ctx.hasUI` is false, so the minimal command just notifies with the raw text.

In [5]:
print("Minimal command in print mode:")
print(ui["noUI"]["printed"][:200])
print()
pinb.show_checks([
    pinb.check("no UI: minimal command runs and returns raw argument text",
               "status now" in ui["minimalCommand"]["lastNotification"], "raw text received"),
])

Minimal command in print mode:
minimal command in print mode (no console.log output)

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------  -----------------
PASS  no UI: minimal command runs and returns raw argument text  raw text received

1/1 assertions held.


## Custom screen: `ctx.mode === "tui"` is the guard

`/guard-blocks` checks `ctx.mode !== "tui"` and refuses with an error in RPC
mode. A custom screen exists **only** where a real terminal is attached.

In [6]:
print(pinb.md_table(
    ["mode", "result"],
    [["RPC (mode=ui)", ui["rpcCustomScreen"]["lastNotification"] and ui["rpcCustomScreen"]["lastNotification"].get("message", "")]],
))

| mode | result |
|---|---|
| RPC (mode=ui) | /guard-blocks requires interactive mode |


In [7]:
pinb.show_checks([
    pinb.check("RPC mode: custom screen is refused, not attempted",
               ui["rpcCustomScreen"]["lastNotification"] and ui["rpcCustomScreen"]["lastNotification"].get("type") == "error" and "interactive mode" in ui["rpcCustomScreen"]["lastNotification"].get("message", ""),
               "refused with error"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------  ------------------
PASS  RPC mode: custom screen is refused, not attempted  refused with error

1/1 assertions held.


## Minimal command: receives raw argument text

`/guard status now` — no parsing, no completions, just the raw string.

In [8]:
print(pinb.md_table(
    ["mode", "received"],
    [["RPC", ui["minimalCommand"]["lastNotification"]]],
))

| mode | received |
|---|---|
| RPC | guard received: "status now" |


In [9]:
pinb.show_checks([
    pinb.check("minimal command receives raw argument text",
               ui["minimalCommand"]["lastNotification"] and "status now" in ui["minimalCommand"]["lastNotification"], "raw text received"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------  -----------------
PASS  minimal command receives raw argument text  raw text received

1/1 assertions held.


## Component API: every line fits the width in display columns

The `BlockListComponent` renders at three widths. Every line must fit, measured
by `visibleWidth` (which handles wide characters correctly). Output is cached by
width and invalidated explicitly.

In [10]:
comp = ui["component"]
for width in [20, 40, 80]:
    r = comp["widths"][str(width)]
    print(f"Width {width}: all_fit={r["allFit"]}, lines={len(r["lines"])}")
    for line in r["lines"]:
        print(f"  {line}")
print()
print(f"Cache: same width returns cached array = {comp["sameCache"]}")
print(f"Cache: invalidate() recomputes = {comp["afterInvalidate"]}")
print(f"Completion callback: q and Escape close = {comp["closed"]}")

Width 20: all_fit=True, lines=6
  Blocked calls
  
  write: path match...
  edit: 路径匹配 .e...
  
  press q or Escape...
Width 40: all_fit=True, lines=6
  Blocked calls
  
  write: path matches a very long path ...
  edit: 路径匹配 .env 受保护的文件 and ...
  
  press q or Escape to close
Width 80: all_fit=True, lines=6
  Blocked calls
  
  write: path matches a very long path that will not fit in twenty columns
  edit: 路径匹配 .env 受保护的文件 and more
  
  press q or Escape to close

Cache: same width returns cached array = True
Cache: invalidate() recomputes = True
Completion callback: q and Escape close = 2


In [11]:
pinb.show_checks([
    pinb.check("width 20: all lines fit in display columns",
               comp["widths"]["20"]["allFit"], "all fit"),
    pinb.check("width 40: all lines fit",
               comp["widths"]["40"]["allFit"], "all fit"),
    pinb.check("width 80: all lines fit",
               comp["widths"]["80"]["allFit"], "all fit"),
    pinb.check("cache: same width returns cached array",
               comp["sameCache"], "cached"),
    pinb.check("invalidate() recomputes",
               comp["afterInvalidate"], "recomputed"),
    pinb.check("q and Escape call completion callback",
               comp["closed"] == 2, "two closes"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------  ----------
PASS  width 20: all lines fit in display columns  all fit
PASS  width 40: all lines fit  all fit
PASS  width 80: all lines fit  all fit
PASS  cache: same width returns cached array  cached
PASS  invalidate() recomputes  recomputed
PASS  q and Escape call completion callback  two closes

6/6 assertions held.


## The chapter's own tests

All seven tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [12]:
PATTERNS = 'ch20-ui/ui.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch20-ui/ui.test.ts', show="component")

$ node --test ch20-ui/ui.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 955 ms

  tests matching 'component':
    ok   component: every line fits the width, measured in columns, including wide characters
    ok   component: output is cached by width and recomputed after invalidate()
    ok   component: it finishes through the completion callback on q or Escape


## Your turn: predict, then run

**Predict first.** What happens if a component's `render(width)` returns a line
that is wider than `width` — does Pi truncate, wrap, or crash?

**Then change one input.** The theme methods `fg(color, text)` and `bold(text)`
are passed in. What happens if the theme returns ANSI escape codes — does
`visibleWidth` count them or strip them?

**Predict the boundary.** The chapter says "the custom screen and entry renderer
were not drawn." What would it take to actually render them in a test — a real
TTY, or can the pi-tui component be tested headless?

In [13]:
print("Predictions:")
print("  1. Line wider than width: visibleWidth is checked in the test; component must truncate.")
print("  2. ANSI codes: visibleWidth strips them — it counts display columns, not bytes.")
print("  3. Headless render: pi-tui components are pure functions (render -> string[]), so they CAN be tested headless.")
print()
print("Observed above:")
print(f"  RPC select: {ui["withUI"]["selectTitle"]}")
print(f"  Minimal command in print mode: works without console.log")
print(f"  RPC custom screen refused: {ui["rpcCustomScreen"]["lastNotification"] and ui["rpcCustomScreen"]["lastNotification"].get("type") == "error"}")
print(f"  Component widths all fit: 20={comp["widths"]["20"]["allFit"]}, 40={comp["widths"]["40"]["allFit"]}, 80={comp["widths"]["80"]["allFit"]}")
print()
assert ui["withUI"]["selectTitle"] == "Path guard"
assert ui["noUI"]["printed"] == "minimal command in print mode (no console.log output)"
assert ui["rpcCustomScreen"]["lastNotification"] and ui["rpcCustomScreen"]["lastNotification"].get("type") == "error"
assert comp["widths"]["20"]["allFit"] and comp["widths"]["40"]["allFit"] and comp["widths"]["80"]["allFit"]
print("held: hasUI vs mode===tui are different guards; component API uses display columns")

Predictions:
  1. Line wider than width: visibleWidth is checked in the test; component must truncate.
  2. ANSI codes: visibleWidth strips them — it counts display columns, not bytes.
  3. Headless render: pi-tui components are pure functions (render -> string[]), so they CAN be tested headless.

Observed above:
  RPC select: Path guard
  Minimal command in print mode: works without console.log
  RPC custom screen refused: True
  Component widths all fit: 20=True, 40=True, 80=True

held: hasUI vs mode===tui are different guards; component API uses display columns


## Interpretation

A command has three presentation modes, and the chapter shows the boundaries:

| Mode | `ctx.hasUI` | `ctx.mode` | What works |
|---|---|---|---|
| Print / JSON | false | `"print"` / `"json"` | `console.log`, `ctx.ui.notify` (prints) |
| RPC | true | `"ui"` | `ctx.ui.select`, `ctx.ui.confirm`, `ctx.ui.notify` |
| TUI | true | `"tui"` | **All of the above** + `ctx.ui.custom` (custom screens), entry renderers |

The component API:
* `render(width: number): string[]` — returns lines that fit in `width` **display columns**
* `handleInput(data: string): void` — receives keystrokes; `q` and `Escape` finish
* `invalidate(): void` — clears the width cache

The practical rules:
* **Degrade gracefully** — check `ctx.hasUI` for dialogs, `ctx.mode === "tui"` for custom screens.
* **Use `visibleWidth`, not `string.length`** — wide characters (CJK, emoji) count as 2 columns.
* **Cache by width** — `invalidate()` only when the data changes, not on every keystroke.
* **Custom screens are TUI-only** — they are an escape hatch for complex UIs; most commands don't need them.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with the chapter's guard-ui and guard-minimal extensions, plus direct component tests. No terminal is attached; RPC mode is used where a UI is needed.

**Evidence scope:** `in_process_runtime`. The command modes, component API, and `hasUI`/`mode` distinction are Pi's.

### What was read or run

- **Ran** `drivers/ch20-ui.ts`: four real sessions (RPC, print, RPC custom screen, minimal) plus direct `BlockListComponent` tests at three widths.
- **Ran** `ch20-ui/ui.test.ts` (7 tests).
- **Read** `examples/evidence.json`, chapter 20 rows (7 claims).

### Limitations

- **The custom screen and entry renderer were not drawn** — they compile against the pinned declarations and the component is tested directly (`ch20-lim1`).
- **No terminal rendering is exercised** — this is a headless run. Layer 1–2 checks (Shift+Enter, Kitty keyboard, tmux passthrough) are in chapter 44 (`ch20-lim2`).
- **`registerToolRenderer` is documented-not-run** by the suite (`ch17-lim3`).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
